In [ ]:
import pandas as pd
from src.data import DATA_DIR_INTERIM

from src.io import load_qrels_from_path, read_metadata
from topic_gen.evaluate import MetaExperiment
from topic_gen.evaluate.io import load_from_irds
from topic_gen.evaluate.measures_agreement import CohenKappa, AreaUnderReceiver, MeanAverageError
from topic_gen.evaluate.measures_agreement_multiple import LabelDistribution

from topic_gen.evaluate.utils import QrelsTransformer

from topic_gen import logger
logger.setLevel("DEBUG")

# Prerequisite
#### Can we reproduce the qrels quality by Thomas et al. with open models?
Yes! Open models outperform the results on GPT 4.1

In [ ]:
BASE_DIR = DATA_DIR_INTERIM / "robust" / "qrels-reference"
experiments = load_qrels_from_path(
    BASE_DIR, binarize_qrels=1, drop_relevance_values=999)

[topic_gen] [WARNING] (io.py:108) Qrels for result 2025-11-30_11:01:37 is empty after processing, skipping...
[topic_gen] [WARNING] (io.py:108) Qrels for result 2025-11-30_12:12:38 is empty after processing, skipping...


In [ ]:
baseline = load_from_irds("disks45/nocr/trec-robust-2004", binarize_qrels=1)
baseline.qrels = QrelsTransformer.drop_relevance(
    baseline.qrels, drop_values=999)

In [ ]:
meta_exp = MetaExperiment(
    experiments=experiments,
    baseline=baseline,
    measures=[CohenKappa(), AreaUnderReceiver(), MeanAverageError(), LabelDistribution()],
    filter_qrels=True
)

In [ ]:
res = meta_exp.evaluate()

In [ ]:
metadata = read_metadata(BASE_DIR)

In [ ]:
df = pd.DataFrame(res)
missing = (
    df[["name", "missing_qrels"]].drop_duplicates(
    ).set_index("name").to_dict()["missing_qrels"]
)

df = df.pivot(index="name", columns="measure", values="value").reset_index()

df = df.merge(metadata, left_on="name", right_on="date")
df["missing"] = df["name"].map(missing)
df["missing"] = df["missing"] - 308459

In [ ]:
df[["name", "model", "prompt", "CohenKappa", "MeanAverageError",
    "AreaUnderReceiver", "missing"]].round(2)

,name,model,prompt,CohenKappa,MeanAverageError,AreaUnderReceiver,missing
0,2025-11-26_11:17:24,gpt-4.1,-DNA-zero-shot,0.64,0.17,0.81,0
1,2025-11-27_06:56:22,Qwen3-30B,-DNA-zero-shot,0.76,0.10,0.90,21
2,2025-11-28_13:57:38,GPT-OSS-120B,-DNA-zero-shot,0.71,0.14,0.84,315
3,2025-11-28_15:17:58,GPT-OSS-20B,-DNA-zero-shot,0.64,0.18,0.82,8
4,2025-11-28_15:28:17,Llama3.1-8B,-DNA-zero-shot,0.63,0.16,0.82,1515
5,2025-11-28_17:37:23,GPT-OSS-120B,-DNA-zero-shot,0.72,0.14,0.85,0
6,2025-11-30_11:47:39,Llama3.1-70B,-DNA-zero-shot,0.77,0.10,0.88,18
7,2025-11-30_12:09:16,Deepseek-V3.2,-DNA-zero-shot,0.72,0.13,0.85,20
8,2025-11-30_12:28:35,Qwen3-14B,-DNA-zero-shot,0.72,0.13,0.85,32


In [ ]:
def to_table(df: pd.DataFrame) -> pd.DataFrame:
    table = df.copy()
    table = table[
        ["model", "CohenKappa", "MeanAverageError", "AreaUnderReceiver", "missing"]
    ]
    table.rename(
        columns={
            "CohenKappa": "Cohen $\kappa$",
            "MeanAverageError": "MAE",
            "AreaUnderReceiver": "AUC",
            "model": "Model",
            "missing": "Missing",
        },
        inplace=True,
    )

    table["Model"].replace(
        {
            "qwen3-14B-no-think": "Qwen3-14B",
            "qwen3-30B-no-think": "Qwen3-30B",
            "deepseek-V3.2": "DeepSeek V3.2",
            "gpt-4.1": "GPT-4.1",
            "gpt-oss-120B": "GPT-OSS-120B",
            "gpt-oss-20B": "GPT-OSS-20B",
            "llama3-1-8B-instruct": "LLaMA-3.1-8B-instruct",
            "llama3-1-70B_instruct_q8_0_MT1000_ollama": "LLaMA-3.1-70B-instruct",
        },
        inplace=True,
    )
    table.to_latex(
        "../publication/paper/tables/pre-requisites-2.tex", index=False)

In [ ]:
# to_table(df)

### Do LLM judges benefit from additional information in the topic?
The original TREC topics are judged by an LLM but one or two topic components are masked. For example, the prompt `-DNA-zero-shot-masked-title` only contains the **description** and **narrative** of the TREC topic. 

- Generally the differences are low
- Masking the title outperforms the full topic
- More context seems to be better

In [ ]:
BASE_DIR = DATA_DIR_INTERIM / "robust" / "qrels-topics-masked"
experiments = load_qrels_from_path(
    BASE_DIR, binarize_qrels=1, drop_relevance_values=999)

[topic_gen] [WARNING] (io.py:108) Qrels for result 2025-12-08_07:00:48 is empty after processing, skipping...
[topic_gen] [WARNING] (io.py:108) Qrels for result 2025-12-08_07:07:17 is empty after processing, skipping...
[topic_gen] [WARNING] (io.py:108) Qrels for result 2025-12-08_07:09:21 is empty after processing, skipping...
[topic_gen] [WARNING] (io.py:108) Qrels for result 2025-12-09_01:00:54 is empty after processing, skipping...


Error loading experiment from 2025-12-08_07:09:33: [Errno 2] No such file or directory: '/workspaces/conf26-generating-topics/data/interim/robust/qrels-topics-masked/2025-12-08_07:09:33/qrels.csv.gz'


In [ ]:
baseline = load_from_irds("disks45/nocr/trec-robust-2004", binarize_qrels=1)
baseline.qrels = QrelsTransformer.drop_relevance(
    baseline.qrels, drop_values=999)

In [ ]:
meta_exp = MetaExperiment(
    experiments=experiments,
    baseline=baseline,
    measures=[CohenKappa(), AreaUnderReceiver(), MeanAverageError(), LabelDistribution()],
    filter_qrels=True
)

In [ ]:
res = meta_exp.evaluate()

In [ ]:
metadata = read_metadata(BASE_DIR)

[topic_gen] [WARNING] (io.py:47) Metadata not found for result 2025-12-08_07:09:33, skipping...


In [ ]:
df = pd.DataFrame(res)
# df["score"] = df.apply(format_score, axis=1)
missing = (
    df[["name", "missing_qrels"]].drop_duplicates(
    ).set_index("name").to_dict()["missing_qrels"]
)

df = df.pivot(index="name", columns="measure", values="value").reset_index()
df = df.merge(metadata, left_on="name", right_on="date")

df["missing"] = df["name"].map(missing)
df["missing"] = df["missing"] - 308459

partial_annotation_prompts = [
    "-DNA-zero-shot",
    "-DNA-zero-shot-masked-title",
    "-DNA-zero-shot-masked-description",
    "-DNA-zero-shot-masked-narrative",
    "-DNA-zero-shot-masked-title-description",
    "-DNA-zero-shot-masked-title-narrative",
    "-DNA-zero-shot-masked-description-narrative",
]
df["prompt"] = pd.Categorical(df["prompt"], partial_annotation_prompts)
df.sort_values(by=["model", "prompt"])[
    ["model", "prompt", "CohenKappa", "MeanAverageError",
        "AreaUnderReceiver", "missing", "label_dist(0)", "label_dist(1)"]
]

,model,prompt,CohenKappa,MeanAverageError,AreaUnderReceiver,missing
14,GPT-OSS-120B,-DNA-zero-shot-masked-title,0.704229,0.141647,0.842270,0
10,GPT-OSS-120B,-DNA-zero-shot-masked-description,0.677359,0.155202,0.829735,0
11,GPT-OSS-120B,-DNA-zero-shot-masked-narrative,0.679708,0.154185,0.830976,0
12,GPT-OSS-120B,-DNA-zero-shot-masked-title-description,0.626208,0.180956,0.805692,0
13,GPT-OSS-120B,-DNA-zero-shot-masked-title-narrative,0.704134,0.141308,0.841940,0
9,GPT-OSS-120B,-DNA-zero-shot-masked-description-narrative,0.704007,0.137242,0.841245,0
8,Llama3.1-70B,-DNA-zero-shot-masked-description-narrative,0.706480,0.125085,0.867056,17
1,Qwen3-30B,-DNA-zero-shot,0.758615,0.102730,0.896110,21
7,Qwen3-30B,-DNA-zero-shot-masked-title,0.791541,0.089880,0.906957,36
3,Qwen3-30B,-DNA-zero-shot-masked-description,0.727497,0.115962,0.879704,19
